# Ejercicio de Feedback 1 - Juan Manuel

**Objetivo:**  Implementar un programa que lea un texto, realice un resumen, determine el sentimiento general y traduzca el texto al inglés.

### Modelos utilizados (Hugging Face)

| Tarea | Modelo |
|---|---|
| Resumen | `mrm8488/bert2bert_shared-spanish-finetuned-summarization` |
| Sentimiento | `nlptown/bert-base-multilingual-uncased-sentiment` |
| Traducción ES→EN | `Helsinki-NLP/opus-mt-es-en` |

## Dependencias e importación de librerías

Para la realización de este ejercicio, se creó un entorno en Conda, puesto que el entorno virtual que tenía para el resto de las prácticas no servía al tener la última versión de Python, considerada no estable para este tipo de tareas. En este entorno instalé lo siguiente: `transformers==4.30.2`, `torch` y `sentencepiece`.

In [1]:
# Las tres librerías instaladas son:
#   - transformers >= 4.30.2  (pipelines de HuggingFace)
#   - torch                   (backend de inferencia neuronal)
#   - sentencepiece           (tokenizador para Helsinki-NLP/opus-mt)
import sys
import transformers, torch
from transformers import pipeline, MarianMTModel, MarianTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# Imprimo versión de Python utilizada, transformers y torch
print(f"\n→ Python:          {sys.version.split()[0]}")
print(f"→ transformers:    {transformers.__version__}")
print(f"→ torch:           {torch.__version__}")


→ Python:          3.11.15
→ transformers:    4.30.2
→ torch:           2.10.0+cu128


## 1. Lectura del texto dado como entrada

El texto se carga desde `texto.txt`, que se encuentra en el mismo directorio.

In [2]:
try:
    with open("./texto.txt", "r", encoding="utf-8") as f:
        texto_original = f.read().strip()
except FileNotFoundError:
    print("texto.txt no encontrado")

# Almaceno el número de palabras y caracteres para poder hacer un buen resumen que se ajuste a la extensión del mismo
num_palabras = len(texto_original.split())
num_chars    = len(texto_original)
print(texto_original)

La tecnología ha revolucionado prácticamente todos los aspectos de nuestras vidas, desde cómo nos
comunicamos hasta cómo realizamos nuestras tareas diarias, cambiando también la manera en que
interactuamos con el mundo que nos rodea. Su impacto es evidente en áreas tan diversas como la
educación, la salud, el transporte, y el comercio, mejorando la eficiencia, la accesibilidad y la
conectividad a niveles nunca antes imaginados. Sin embargo, a pesar de los innumerables beneficios que
nos brinda, la tecnología también trae consigo desafíos complejos y significativos que no podemos
ignorar.
Uno de los retos más destacados es la brecha digital, que exacerba las desigualdades sociales y
económicas entre quienes tienen acceso a las tecnologías y quienes no. Esta disparidad limita las
oportunidades educativas, laborales y sociales de muchas personas en diversas partes del mundo,
dejando atrás a comunidades enteras en el desarrollo tecnológico. Además, la expansión de la tecnología
ha plantead

## 2. Generación de un resumen del texto

- Para realizar el resumen del texto he escogido el modelo español **`bert2bert_shared-spanish-finetuned-summarization`**, que se trata de un modelo fine-tuned utilizando MLSUM, el primer dataset de resúmenes multilingües a gran escala.

- La principal ventaja es que a diferencia de Bart, que fue usado en las prácticas de la asignatura como ejemplo, está entrenado en inglés, mientras que este modelo entiende español nativo y está entrenado con fuentes reales como noticias, lo cual es una gran ventaja ya que el texto de ejemplo a resumir se trata de un texto informativo.

In [ ]:
# Cargo el modelo de resumen
nombre_modelo = "mrm8488/bert2bert_shared-spanish-finetuned-summarization"

tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
model = AutoModelForSeq2SeqLM.from_pretrained(nombre_modelo)

#  bert2bert_shared-spanish-finetuned-summarization
resumidor = pipeline(
    "summarization",
    model=model,
    tokenizer=tokenizer,
    device=-1      # -1 = CPU
)

In [ ]:
def generar_resumen(texto_en: str) -> str:
    """
    Genera un resumen usando mrm8488/bert2bert_shared-spanish-finetuned-summarization.

    Args:
        texto_en   : Texto en inglés a resumir.

    Returns:
        Cadena de texto con el resumen.
    """
    din_max = min(120, int(num_palabras * 0.4)) # Máximo el 40% del texto
    din_min = min(50, int(num_palabras * 0.2))
    resultado = resumidor(
        texto_en,
        max_length=din_max,
        min_length=din_min,
        do_sample=True,          # Generación con muestreo (no determinista)
        top_p=0.92,              # Filtro de palabras con sentido
        num_beams=4,             # Búsqueda por haz para mejor coherencia
        no_repeat_ngram_size=3,  # Evita bucles de palabras
        length_penalty=1.5,      # Favorece frases bien construidas
        repetition_penalty=1.2,  # Penalización suave de repetición
        truncation=True
    )
    # Devuelvo el resumen del texto
    return resultado[0]["summary_text"]

In [10]:
resumen_en = generar_resumen(texto_original)
print(resumen_en + ".")

La brecha digital, que exacerba las desigualdades sociales y económicas entre quienes tienen acceso a las tecnologías y quienes no pueden acceder a las nuevas tecnologías, se ha convertido en una fuente de desigualdad o amenaza para la mayor cantidad de personas posibles en el mundo que nos rodea.


## 3. Análisis del sentimiento general del texto

Para el análisis de sentimiento del texto se usó **`nlptown/bert-base-multilingual-uncased-sentiment`**, entrenado en reseñas en 6 idiomas (inglés, alemán, neerlandés, francés, **español** e italiano) y que clasifica el texto en 5 niveles:

| Estrellas | Categoría |
|:---:|---|
| ⭐ 1-2 | Negativo |
| ⭐⭐⭐ 3 | Neutral |
| ⭐⭐⭐⭐-⭐⭐⭐⭐⭐ 4-5 | Positivo |

In [ ]:
analizador = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=-1
)

In [ ]:
def analizar_sentimiento(texto: str) -> dict:
    """
    Analiza el sentimiento del texto usando BERT multilingüe.

    El modelo acepta hasta 512 tokens; textos más largos se truncan
    automáticamente (el inicio del texto fija el tono general).

    Args:
        texto: Texto en español a analizar.

    Returns:
        Diccionario con: categoria, confianza, etiqueta_raw.
    """
    resultado     = analizador(texto, truncation=True, max_length=512)
    etiqueta_raw  = resultado[0]["label"]   
    confianza     = resultado[0]["score"]           # Probabilidad 0-1
    estrellas     = int(etiqueta_raw.split()[0])    # Número de estrellas

    # Mapeo las estrellas para clasificar cada sentimiento
    if estrellas <= 2:
        categoria = "Negativo"
    elif estrellas == 3:
        categoria = "Neutral"
    else:
        categoria = "Positivo"

    return {
        "categoria"   : f"{categoria}",
        "confianza"   : confianza,
        "etiqueta_raw": etiqueta_raw
    }

In [13]:
resultado_sentimiento = analizar_sentimiento(texto_original)

print(f"  Sentimiento detectado : {resultado_sentimiento['categoria']}")
print(f"  Confianza del modelo  : {resultado_sentimiento['confianza']:.2%}")
print(f"  Etiqueta original     : {resultado_sentimiento['etiqueta_raw']}")

  Sentimiento detectado : Positivo
  Confianza del modelo  : 50.16%
  Etiqueta original     : 5 stars


Vistos estos resultados se puede observar que el modelo ha identificado el texto como esperanzador, por lo que lo ha asociado con la categoría de texto "positivo", pero con una confianza baja, puesto que existen numerosos elementos en el texto como "amenaza" o "desigualdad" que hacen que la clasificación sea más incierta.

## 4. Traducción del texto original al inglés

Para la traducción del texto del español al inglés, se ha utilizado el modelo **`Helsinki-NLP/opus-mt-es-en`**.

In [ ]:
MODELO_TRAD    = "Helsinki-NLP/opus-mt-es-en"
tokenizer_trad = MarianTokenizer.from_pretrained(MODELO_TRAD)
modelo_trad    = MarianMTModel.from_pretrained(MODELO_TRAD)

In [15]:
def traducir_es_en(texto: str) -> str:
    # Limpio saltos de línea manteniendo párrafos reales
    texto_limpio = texto.replace("\n", " ")
    # Elimino espacios dobles que puedan confundir al modelo
    while "  " in texto_limpio:
        texto_limpio = texto_limpio.replace("  ", " ")

    # Como el texto es pequeño cabe sin necesidad de trocearlo
    inputs = tokenizer_trad(
        texto_limpio, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=512
    )
    
    # Uso num_beams=4 para máxima calidad, ya que usa beam search para obtener las mejores palabras en inglés
    tokens_out = modelo_trad.generate(**inputs, num_beams=4, max_length=512)
    return tokenizer_trad.decode(tokens_out[0], skip_special_tokens=True)

In [16]:
texto_en_ingles = traducir_es_en(texto_original)
print(texto_en_ingles)

Technology has revolutionized virtually every aspect of our lives, from how we communicate to how we perform our daily tasks, also changing the way we interact with the world around us. Its impact is evident in areas as diverse as education, health, transport, and trade, improving efficiency, accessibility and connectivity at levels never before imagined. However, despite the countless benefits it brings us, technology also brings with it complex and significant challenges that we cannot ignore. One of the most important challenges is the digital divide, which exacerbates social and economic inequalities between those who have access to technologies and those who do not. This disparity limits the educational, employment and social opportunities of many people in various parts of the world, leaving entire communities behind in technological development. In addition, the expansion of technology has raised serious concerns about privacy and the security of personal data, since much of our